In [0]:
# Imports
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
# Read CSV from Unity Catalog into Spark DataFrame
raw_df = spark.read.csv(
    "/Volumes/students_data/diarmuid-gallagher/technologist-volume/taxi_data.csv",
    header=True,
    inferSchema=True,
    mode="DROPMALFORMED" # Drop malformed rows
)

In [0]:
# Verify file exists
display(
    dbutils.fs.ls(
        "/Volumes/students_data/diarmuid-gallagher/technologist-volume/taxi_data.csv"
    )
)
# Check dataset loaded
display(raw_df)

In [0]:
# Create Bronze DataFrame - add timestamp & source_file
bronze_df = (
    raw_df
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_source_file", lit("/Volumes/students_data/diarmuid-gallagher/technologist-volume/taxi_data.csv"))
)

In [0]:
# Create Bronze Delta Table
if not spark.catalog.tableExists("bronze_taxi"):

    bronze_df.write \
        .format("delta") \
        .option("delta.columnMapping.mode", "name") \
        .saveAsTable("bronze_taxi")

In [0]:
%sql
-- Return row count
SELECT COUNT(*) FROM bronze_taxi;


In [0]:
%sql
--  View Bronze Taxi Table
SELECT *
FROM bronze_taxi
LIMIT 5;

In [0]:
%sql
-- Verify timestamp and source file column added 
SELECT
    _ingest_timestamp,
    _source_file
FROM bronze_taxi
LIMIT 10;

In [0]:
%sql
-- Output Table Schema
DESCRIBE bronze_taxi;